In [32]:
import os
import sys
import glob
import pickle
import numpy as np
import matplotlib.pyplot as plt
import h5py
from astropy.io import fits
import csv

from astropy.cosmology import FlatLambdaCDM
import astropy.units as u
import astropy.constants as const
from astropy.coordinates import SkyCoord

# funky plot packages
from astropy.convolution import convolve, Gaussian2DKernel, Tophat2DKernel
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import SymLogNorm

sys.path.insert(0, os.getcwd() + '/stacking/')
import lim_stacker as st

from multiprocessing import Pool
from functools import partial

In [30]:
currentdir = os.getcwd() + '/';

In [33]:
currentdir

'/home/finneas/Documents/COMAP/bootstrapping (2)/'

In [34]:
 def n_stacks_multi(seed, savepath, goalnobj, centfreq, maplist, catlist, i):

    params = st.parameters()
    #params.saveplots = False
    #params.savedata = False
    params.savepath = currentdir + savepath
    params.verbose = True
    params.bootverbose = True
    params.chanwidth = 0.03125 #***

    params.chanmeanfilter = False
    params.lowmodefilter = False
    params.rotate = True

    #params.plotspace = False
    #params.plotfreq = False
    #params.plotcubelet = False
    params.returncutlist = False
    params.usefeed = False
    params.physicalspace = False

    params.freqstackwidth = 4
    params.spacestackwidth = 2
    params.xwidth = 5
    params.ywidth = 5
    params.itersavefile = params.savepath+'random_stacker_iter_output.csv'
    params.nitersavefile = params.savepath + 'rand_stacker_output.npz'
    params.create_sensmap_bootstrap(currentdir + 'oii_based_randoms/', cat=True)

    params.centfreq = centfreq
    params.goalnumcutouts = goalnobj

    #params.hx_random_path = '/mn/stornext/d16/cmbco/comap/delaney/stacks/hetdex/HETDEX_sensitivity_randoms/hetdex_recovered_randoms_cutz_250729.csv'

    Tvals = []
    Trmsvals = []

    if params.bootverbose:
        if i % 1 == 0:
            print('iteration {}'.format(i))

    # set up an rng for the offsets
    offrng = np.random.default_rng(seed+i)

    outcube  = st.offset_and_stack(maplist, catlist, params, offrng, method=method, return_cube=True)
    outvals = [outcube.linelum, outcube.dlinelum, outcube.ncutouts]
     
    with open(params.itersavefile, 'a') as csvfile:
        w = csv.writer(csvfile)
        w.writerow(outvals)
        

    return outvals

In [35]:
# standard COMAP cosmology
cosmo = FlatLambdaCDM(H0=70*u.km / (u.Mpc*u.s), Om0=0.286, Ob0=0.047)

s12stackmapfiles = [currentdir+'comap_data/co2_s12_v5_take6_n5_subtr_exper.h5',
                    currentdir+'comap_data/co6_s12_v5_take6_n5_subtr_exper.h5',
                    currentdir+'comap_data/co7_s12_v5_take6_n5_subtr_exper.h5']

In [42]:
##### Initial setup garbaj. Change this for different surveys
mapfiles = s12stackmapfiles
#method = 'offset' # 'offset' for qsos, for LAEs,use 'catalogue'

#####


""" PARAMETERS """
params = st.parameters()
#params.saveplots = False

#params.plotspace = False
#params.plotfreq = False
#params.plotcubelet = False
params.savepath = 'VERY_DIFFERENT_SAVEPATH'


params.returncutlist = False
params.physicalspace = False
params.usefeed = False
params.itersave = True
params.itersavestep = 10
params.freqwidth = 3 #**
params.xwidth = 5
params.ywidth = 5
params.chanwidth = 0.03125 #**
params.create_sensmap_bootstrap(currentdir+'oii_based_randoms/', cat=True)


# random seed for offsets
params.bootstrapseed = 12345

params.verbose = True

params.make_output_pathnames()

In [37]:
#goalnobj_channels = [[387, 0, 55],[385, 0, 53],[382, 0, 55],[381, 0, 55],[373, 0, 54],[370, 0, 55],[369, 0, 55],[370, 0, 55],[373, 0, 55],[371, 0, 54],[369, 0, 56]] # QSOs 100ghz - 101ghz
#goalnobj_channels = [5450, 175, 7197],[5431, 180, 7184],[5397, 177, 7155],[5359, 168, 7115],[5336, 161, 7060],[5279, 153, 7011],[5248, 159, 6986],[5206, 151, 6933],[5182, 153, 6907],[5136, 155, 6895], [5126, 157, 6881]

# offset
#goalnobj_channels = [[377,0,55],[370,0,54],[371,0,55],[367,0,54],[372,0,55],[372,0,54],[369,0,56],[367,0,57],[368,0,57],[362,0,57],[359,0,58]]
#goalnobj_channels = [[1384,18,1565],[1400,20,1582],[1417,18,1590],[1425,15,1599],[1439,17,1605],[1452,20,1617],[1463,18,1628],[1469,17,1640],[1459,18,1647],[1462,18,1639],[1462,16,1644]]; # OFFSETTED

In [38]:

def bootstrap_over_channels(goalnobj_channels, centfreq_channels, num_bootstraps, sstr):
    for index in range(len(goalnobj_channels)):
        
        goalnobj = goalnobj_channels[index]
        centfreq = centfreq_channels[index]
    
        ### Create directory for each step
        savepath = sstr+'_'+str(index)+'/'
        params.savepath = currentdir+savepath
        
        if not os.path.exists(params.savepath):
            os.makedirs(params.savepath)
    
        params.itersavefile = params.savepath+'random_stacker_iter_output.csv'
        params.nitersavefile = params.savepath + 'rand_stacker_output.npz'
        itersavefile = params.itersavefile
        
        ### Setup stuff
        
        maplist, catlist = st.setup(mapfiles, catfile, params)
        print(len(catlist[0].z))
        
        # offset
        offset = 200*u.km/u.s
        
        for i in range(len(catlist)):
            dz = offset / (const.c.to(u.km/u.s)) * catlist[i].z
            newz = dz + catlist[i].z
        
            catlist[i].z = newz.value
        
        # initialize the output file
        with open(itersavefile, 'w') as csvfile:
            w = csv.writer(csvfile)
            w.writerow(['linelum', 'dlinelum'])
        
        # run the actual stack purely to see how many cutouts you're going to need for each bootstrap
        # outs = st.stacker(maplist, catlist, params)
        
        # actcatidx = outs[-1]
        
        # set the goal numbers of cutouts
        params.goalnumcutouts = goalnobj
        print(params.goalnumcutouts)
        
        # set up an rng for the offsets
        offrng = np.random.default_rng(params.bootstrapseed)
        
        # play with the output that's printed so you don't get every cutout for every stack
        if params.verbose:
            params.bootverbose = True
            params.verbose = False
        else:
            params.bootverbose = False
        
        print('Done Setup')
    
        ### Do the actual bootstrapping
    
        p = Pool(processes=10)
        stackvals = p.map(partial(n_stacks_multi, 1234 + num_bootstraps*index, savepath, goalnobj, centfreq, maplist, catlist), range(num_bootstraps))
        
        stackvals = np.stack(stackvals)
        
        csvfile.close()
        
        np.savez(params.nitersavefile, T=stackvals[:,0], rms=stackvals[:,1])

In [41]:
freqrange = [100.0, 100.1, 100.2, 100.3, 100.4, 100.5, 100.6, 100.7, 100.8, 100.9, 101.0]
method = 'offset';
catfile = currentdir+'catalogs/QSO_cat_iron_cumulative_v0.npz'
bootstrap_over_channels([[377,0,55],[370,0,54],[371,0,55],[367,0,54],[372,0,55],[372,0,54],[369,0,56],[367,0,57],[368,0,57],[362,0,57],[359,0,58]], freqrange, 5, 'output/QSO_100.0-101.0');
method = 'catalogue'; 
catfile = currentdir+'catalogs/SURVEY_HETDEX_LAE_2.39_z_3.44.npz' # used to be combined_HETDEX_noduplicates.npz
bootstrap_over_channels([[1384,18,1565],[1400,20,1582],[1417,18,1590],[1425,15,1599],[1439,17,1605],[1452,20,1617],[1463,18,1628],[1469,17,1640],[1459,18,1647],[1462,18,1639],[1462,16,1644]], freqrange, 10, 'output/LAE_100.0-101.0');

scaling rms
scaling rmsmasking isolated signal pixels


KeyboardInterrupt: 